# Clean-air baselines (R0)

R0 is each MQ sensor's resistance in clean air on this unit. Scores are computed
against it, so a wrong R0 shifts every gas and smoke score. This notebook takes R0 from
a settled stretch of a capture with the same rule the firmware's `baseline` command
uses (median, with spread = (p90 - p10) / median), and compares it with any
`calibration` events the node emitted.

**Procedure.** After warm-up (notebook 01), record 10 minutes of clean air with the fan
in the state it will run in. Optionally run `baseline mq2` and `baseline mq135` on the
console during the capture so both methods can be compared.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # run from tools/calibration, next to fxcal.py
import fxcal
import matplotlib.pyplot as plt

In [ ]:
CAPTURE = None  # e.g. "../../captures/clean-air-2026-10-01.csv"
EVENTS = None   # e.g. "../../captures/clean-air-2026-10-01.events.jsonl"
SETTLED_FROM_S = 600  # device seconds since boot: after the warm-up measured in 01

cap = fxcal.load_capture(CAPTURE) if CAPTURE else fxcal.demo_warmup()
events = fxcal.load_events(EVENTS) if EVENTS else []
if not CAPTURE:
    print("SYNTHETIC DATA. Set CAPTURE to a bench recording before drawing any conclusion.")

In [ ]:
MAX_SPREAD = 0.05  # the firmware default tolerance

for column, sensor in (("mq2_rs", "mq2"), ("mq135_rs", "mq135")):
    ts, vs = fxcal.series(cap, column, t_from=SETTLED_FROM_S)
    if len(vs) < 10:
        print(f"{sensor}: not enough settled samples")
        continue
    r0, spread = fxcal.baseline(vs)
    verdict = "ok" if spread <= MAX_SPREAD else "UNSTABLE, do not use"
    print(f"{sensor}: R0 = {r0:.0f} ohm from {len(vs)} samples, spread {spread:.3%} ({verdict})")
    print(f"    console: r0 {sensor} {r0:.0f}")

for event in events:
    if event.get("type") == "calibration":
        print(f"device {event['sensor']}: R0 {event['r0']} ohm, spread {event['spread']:.3%}, {event['result']}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for column in ("mq2_rs", "mq135_rs"):
    ts, vs = fxcal.series(cap, column, t_from=SETTLED_FROM_S)
    if vs:
        r0, _ = fxcal.baseline(vs)
        ax.plot([t / 60 for t in ts], [v / r0 for v in vs], label=f"{column} / R0")
ax.axhspan(0.95, 1.05, alpha=0.15)
ax.set_xlabel("minutes since power-on")
ax.set_ylabel("Rs / R0")
ax.legend()
plt.show()